# 06 — Feature Engineering para Modelo de Classificação: Banco Ganhou vs Perdeu

## Objetivo do modelo

Criar o primeiro conjunto de features para um modelo de classificação cujo objetivo é prever:

```text
target_banco_ganhou = 1 → banco ganhou / êxito
target_banco_ganhou = 0 → banco perdeu / condenação
```

## O modelo faz sentido?

Sim, a ideia faz sentido, desde que o problema seja definido com cuidado.

Este modelo pode ajudar o jurídico a responder:

1. Quais processos têm maior probabilidade de perda?
2. Quais assuntos/produtos/ações historicamente levaram a mais perdas?
3. Quais casos merecem priorização humana?
4. Quais temas precisam de revisão de tese, documentação ou estratégia de acordo?
5. Quais novas ações têm maior risco jurídico?

## Principal cuidado: leakage

Como o target foi criado a partir de `Situacao` e `Motivo_Encerramento`, essas colunas **não podem entrar como features**.

Também devem ser removidas variáveis que só existem depois do desfecho, como:

- `Dataencerramento`
- `Motivo_Encerramento`
- `desfecho_categoria`
- `target_banco_ganhou`
- `target_banco_perdeu`
- `valor_perdido`
- `valor_ganho`
- resultados/sentenças/condenações/acordos da DeepLegal, dependendo do momento de previsão

## Primeira versão recomendada

Para o primeiro modelo, vamos construir uma ABT com três conjuntos de features:

### 1. Features seguras para baseline

Features que normalmente existem no início ou no cadastro do processo:

- carteira;
- produto;
- ação;
- assunto;
- tipo do processo;
- UF/comarca;
- valor ajuizado;
- data inicial;
- empresa;
- tipo de justiça/processo, se disponível;
- classe/assunto DeepLegal, se disponível antes do resultado.

### 2. Features de gestão interna com cautela

Podem existir após algum tempo de acompanhamento:

- fase do processo;
- dias desde último andamento;
- advogado interno;
- escritório credenciado;
- passível de acordo;
- processo relevante;
- estimativa de perda.

Essas podem ser úteis em um **score atual da carteira**, mas precisam de validação temporal para um modelo realmente preditivo.

### 3. Features proibidas ou de alto risco de leakage

Não usar no primeiro modelo preditivo:

- motivo de encerramento;
- situação;
- data de encerramento;
- resultado final;
- sentença;
- condenação;
- acordo realizado;
- valor de condenação;
- valor de acordo;
- predições externas DeepLegal de procedência/condenação/acordo;
- textos de sentença/tutela/resultado, quando forem posteriores.

## Entrada esperada

Este notebook pode partir de:

```text
data/processed/abt_eda_valorajuizado_target_banco_ganhou.parquet
```

ou de um `df_joined` em memória.

A entrada ideal é a ABT criada no notebook anterior.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 240)

BASE_DIR = Path("..")
PROCESSED_DIR = BASE_DIR / "data" / "processed"
REPORTS_DIR = BASE_DIR / "outputs" / "reports"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

INPUT_FILE = PROCESSED_DIR / "abt_eda_valorajuizado_target_banco_ganhou.parquet"

TARGET_COL = "target_banco_ganhou"
VALUE_COL = "valor_ajuizado"
RANDOM_STATE = 42

print("Setup concluído.")

## 2. Carregar base

A preferência é carregar a ABT criada no notebook anterior:

```text
data/processed/abt_eda_valorajuizado_target_banco_ganhou.parquet
```

Se ela não existir, o notebook tenta usar `df_joined` em memória.

In [ ]:
if INPUT_FILE.exists():
    df = pd.read_parquet(INPUT_FILE)
    print(f"Arquivo carregado: {INPUT_FILE}")
elif "df_joined" in globals():
    df = df_joined.copy()
    print("Usando df_joined em memória.")
else:
    raise FileNotFoundError(
        f"Arquivo não encontrado: {INPUT_FILE}. "
        "Rode antes o notebook 05 ou carregue df_joined em memória."
    )

print(df.shape)
df.head()

## 3. Funções auxiliares

In [ ]:
NULL_STRINGS = {
    "", "null", "nan", "none", "na", "n/a", "-",
    "não informado", "nao informado", "sem informação", "sem informacao"
}

def normalize_text(x):
    if pd.isna(x):
        return None
    x = str(x).strip().lower()
    x = unicodedata.normalize("NFKD", x)
    x = "".join(c for c in x if not unicodedata.combining(c))
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    if x in NULL_STRINGS:
        return None
    return x

def safe_divide(a, b):
    return np.where((b == 0) | pd.isna(b), np.nan, a / b)

def existing_cols(df, cols):
    return [c for c in cols if c in df.columns]

def save_report(df_report, filename):
    path = REPORTS_DIR / filename
    df_report.to_csv(path, index=False)
    print(f"Salvo: {path}")

def parse_sim_nao(x):
    x = normalize_text(x)
    if x in ["sim", "s", "yes", "true", "1"]:
        return 1
    if x in ["nao", "não", "n", "no", "false", "0"]:
        return 0
    return np.nan

def map_estimativa_perda(x):
    x = normalize_text(x)
    if x is None:
        return np.nan
    if "remoto" in x:
        return 0
    if "possivel" in x:
        return 1
    if "provavel" in x:
        return 2
    return np.nan

def winsorize_series(s, lower_q=0.01, upper_q=0.99):
    lo = s.quantile(lower_q)
    hi = s.quantile(upper_q)
    return s.clip(lo, hi)

def clean_colname(name):
    name = str(name)
    name = unicodedata.normalize("NFKD", name)
    name = "".join(c for c in name if not unicodedata.combining(c))
    name = re.sub(r"[^a-zA-Z0-9_]", "_", name)
    name = re.sub(r"_+", "_", name).strip("_").lower()
    return name

## 4. Validar target e remover linhas inválidas

Esta etapa garante que o modelo use apenas linhas com target definido.

In [ ]:
if TARGET_COL not in df.columns:
    raise KeyError(f"Target {TARGET_COL} não encontrado. Rode antes o notebook que cria o target.")

df_model = df.copy()
df_model = df_model[df_model[TARGET_COL].notna()].copy()
df_model[TARGET_COL] = df_model[TARGET_COL].astype(int)
df_model = df_model[df_model[TARGET_COL].isin([0, 1])].copy()

target_summary = df_model.groupby(TARGET_COL).agg(qtd=(TARGET_COL, "size")).reset_index()
target_summary["classe"] = target_summary[TARGET_COL].map({1: "banco_ganhou", 0: "banco_perdeu"})
target_summary["perc"] = target_summary["qtd"] / target_summary["qtd"].sum()

save_report(target_summary, "fe_target_distribution.csv")
target_summary

## 5. Definir colunas com leakage

Essas colunas não devem entrar no modelo de classificação preditiva.

### Leakage direto

São colunas usadas para criar o target ou que representam o resultado do processo:

- `Situacao`
- `Motivo_Encerramento`
- `desfecho_categoria`
- `target_banco_ganhou`
- `target_banco_perdeu`
- `valor_perdido`
- `valor_ganho`

### Leakage temporal

São colunas que provavelmente só são conhecidas depois do encerramento ou depois de eventos decisivos:

- `Dataencerramento`
- `resultado_final_processo_text`
- `resultadojulgamento_tipo`
- `sentenca_tipo`
- `condenacao_valor`
- `valor_do_acordo_valor`
- `valor_arbitrado_valor`
- `resultado_do_recurso_texto`
- `resultado_exec_texto`
- `texto_sentenca_texto`
- `texto_tutela_texto`

### Predições externas

Predições da DeepLegal podem ser úteis em um score operacional, mas para o primeiro modelo próprio elas devem ser removidas para evitar dependência/leakage indireto.

In [ ]:
leakage_exact_cols = [
    TARGET_COL,
    "target_banco_perdeu", "desfecho_categoria", "valor_perdido", "valor_ganho",
    "Situacao", "Motivo_Encerramento", "Dataencerramento",
    "resultado_final_processo_text", "resultadojulgamento_tipo", "sentenca_tipo",
    "condenacao_valor", "valor_do_acordo_valor", "valor_arbitrado_valor",
    "resultado_do_recurso_texto", "resultado_exec_texto",
    "texto_sentenca_texto", "texto_tutela_texto", "data_condenacao_data",
    "sentenca_data", "data_sentenca_execucao_data", "data_do_acordao_data",
    "resultado_acordao", "valores_processo", "transitojulgado_data",
    "data_transito_julgado_data", "data_transito_julgado_extinto_ou_improcedente_data",
    "data_transito_julgado_procedente_ou_parc_procedente_data", "data_arquivado_2a_data",
    "rank", "rank_valor", "rank_percentual", "valor_acumulado", "perc_valor_acumulado",
    "valor_perdido_acumulado", "perc_valor_perdido_acumulado",
]

leakage_patterns = [
    r"^target_", r"^desfecho", r"^valor_perdido", r"^valor_ganho",
    r"resultado", r"sentenca", r"condenacao", r"acordo_prob",
    r"procedente_prob", r"procedente_parc", r"dano_moral_prob", r"provavel_",
    r"texto_sentenca", r"texto_tutela", r"transito", r"arquivado_data",
    r"data_arquivado", r"data_condenacao", r"valor_do_acordo", r"valor_arbitrado",
]

def is_leakage_col(col):
    if col in leakage_exact_cols:
        return True
    c = str(col).lower()
    return any(re.search(pat, c) for pat in leakage_patterns)

leakage_cols = [c for c in df_model.columns if is_leakage_col(c)]

leakage_report = pd.DataFrame({"coluna": leakage_cols, "motivo": "leakage_target_ou_pos_desfecho"})
save_report(leakage_report, "fe_leakage_cols_removed.csv")
print("Qtd colunas leakage removidas:", len(leakage_cols))
leakage_report.head(100)

## 6. Criar features financeiras

Features de valor são importantes porque o valor da causa pode influenciar estratégia, complexidade e risco.

Criamos:

- valor bruto;
- log do valor;
- valor winsorizado;
- log winsorizado;
- faixas de valor;
- flags de alto valor.

In [ ]:
if VALUE_COL not in df_model.columns and "Valorajuizado" in df_model.columns:
    df_model[VALUE_COL] = pd.to_numeric(df_model["Valorajuizado"], errors="coerce")

if VALUE_COL not in df_model.columns:
    raise KeyError(f"Coluna de valor {VALUE_COL} não encontrada.")

df_model[VALUE_COL] = pd.to_numeric(df_model[VALUE_COL], errors="coerce")

df_model["fe_valor_ajuizado"] = df_model[VALUE_COL]
df_model["fe_valor_ajuizado_log1p"] = np.log1p(df_model["fe_valor_ajuizado"].clip(lower=0))
df_model["fe_valor_ajuizado_winsor"] = winsorize_series(df_model["fe_valor_ajuizado"])
df_model["fe_valor_ajuizado_winsor_log1p"] = np.log1p(df_model["fe_valor_ajuizado_winsor"].clip(lower=0))

for q in [0.75, 0.90, 0.95, 0.99]:
    threshold = df_model["fe_valor_ajuizado"].quantile(q)
    suffix = str(int(q * 100))
    df_model[f"fe_flag_valor_acima_p{suffix}"] = (df_model["fe_valor_ajuizado"] >= threshold).astype(int)

bins = [-np.inf, 1000, 5000, 10000, 25000, 50000, 100000, 250000, 500000, 1000000, np.inf]
labels = ["ate_1k", "1k_5k", "5k_10k", "10k_25k", "25k_50k", "50k_100k", "100k_250k", "250k_500k", "500k_1mm", "acima_1mm"]

df_model["fe_faixa_valor_ajuizado"] = pd.cut(df_model["fe_valor_ajuizado"], bins=bins, labels=labels, include_lowest=True).astype(str)

financial_features = [
    "fe_valor_ajuizado", "fe_valor_ajuizado_log1p", "fe_valor_ajuizado_winsor",
    "fe_valor_ajuizado_winsor_log1p", "fe_flag_valor_acima_p75",
    "fe_flag_valor_acima_p90", "fe_flag_valor_acima_p95", "fe_flag_valor_acima_p99",
    "fe_faixa_valor_ajuizado",
]

df_model[financial_features].head()

## 7. Criar features temporais

Features de data inicial geralmente são seguras, pois existem no início.

Criamos:

- ano;
- mês;
- trimestre;
- semestre;
- dia da semana;
- idade do processo até uma data de referência.

Para modelagem real, `data_ref` deve representar a data em que a previsão seria feita. Neste baseline, usamos a data máxima disponível de `Datainicial` apenas como aproximação exploratória.

In [ ]:
date_features = []

if "Datainicial" in df_model.columns:
    df_model["Datainicial"] = pd.to_datetime(df_model["Datainicial"], errors="coerce")
    data_ref = df_model["Datainicial"].max()

    df_model["fe_ano_inicial"] = df_model["Datainicial"].dt.year
    df_model["fe_mes_inicial"] = df_model["Datainicial"].dt.month
    df_model["fe_trimestre_inicial"] = df_model["Datainicial"].dt.quarter
    df_model["fe_semestre_inicial"] = np.where(df_model["fe_mes_inicial"] <= 6, 1, 2)
    df_model["fe_dia_semana_inicial"] = df_model["Datainicial"].dt.dayofweek
    df_model["fe_idade_desde_inicio_ate_ref_dias"] = (data_ref - df_model["Datainicial"]).dt.days

    df_model["fe_faixa_ano_inicial"] = pd.cut(
        df_model["fe_ano_inicial"],
        bins=[-np.inf, 2016, 2018, 2020, 2022, 2024, np.inf],
        labels=["ate_2016", "2017_2018", "2019_2020", "2021_2022", "2023_2024", "apos_2024"]
    ).astype(str)

    date_features = [
        "fe_ano_inicial", "fe_mes_inicial", "fe_trimestre_inicial", "fe_semestre_inicial",
        "fe_dia_semana_inicial", "fe_idade_desde_inicio_ate_ref_dias", "fe_faixa_ano_inicial",
    ]

date_features

## 8. Criar features de gestão interna

Essas features podem ser muito boas, mas exigem cautela.

Algumas talvez só existam depois de certo tempo de acompanhamento. Para o primeiro baseline, podemos manter em um cenário chamado `score_atual`.

Para um modelo realmente preditivo no início do processo, avalie remover:

- `Fasedoprocesso`;
- `Passiveldeacordo`;
- `Processorelevante`;
- `Estimativa_De_Perda`;
- `Qtddias_Ultimoandamento`.

In [ ]:
management_features = []

if "Passiveldeacordo" in df_model.columns:
    df_model["fe_flag_passivel_acordo"] = df_model["Passiveldeacordo"].apply(parse_sim_nao)
    management_features.append("fe_flag_passivel_acordo")

if "Processorelevante" in df_model.columns:
    df_model["fe_flag_processo_relevante"] = df_model["Processorelevante"].apply(parse_sim_nao)
    management_features.append("fe_flag_processo_relevante")

if "Estimativa_De_Perda" in df_model.columns:
    df_model["fe_estimativa_perda_score"] = df_model["Estimativa_De_Perda"].apply(map_estimativa_perda)
    df_model["fe_flag_estimativa_possivel_ou_provavel"] = (df_model["fe_estimativa_perda_score"] >= 1).astype(float)
    df_model.loc[df_model["fe_estimativa_perda_score"].isna(), "fe_flag_estimativa_possivel_ou_provavel"] = np.nan
    management_features += ["fe_estimativa_perda_score", "fe_flag_estimativa_possivel_ou_provavel"]

if "Qtddias_Ultimoandamento" in df_model.columns:
    df_model["fe_qtd_dias_ultimo_andamento"] = pd.to_numeric(df_model["Qtddias_Ultimoandamento"], errors="coerce")
    df_model["fe_flag_sem_andamento_90d"] = (df_model["fe_qtd_dias_ultimo_andamento"] >= 90).astype(float)
    df_model["fe_flag_sem_andamento_180d"] = (df_model["fe_qtd_dias_ultimo_andamento"] >= 180).astype(float)
    df_model["fe_flag_sem_andamento_360d"] = (df_model["fe_qtd_dias_ultimo_andamento"] >= 360).astype(float)
    management_features += [
        "fe_qtd_dias_ultimo_andamento", "fe_flag_sem_andamento_90d",
        "fe_flag_sem_andamento_180d", "fe_flag_sem_andamento_360d",
    ]

management_features

## 9. Normalizar variáveis categóricas

Vamos criar versões normalizadas de campos categóricos importantes.

Isso ajuda modelos e melhora encoders de alta cardinalidade.

In [ ]:
base_categorical_cols = existing_cols(df_model, [
    "Carteira", "Nm_Produto", "Nm_Acao", "Nm_Assunto", "Tipoprocesso", "Uf", "Comarca",
    "Fasedoprocesso", "Credenciado", "Adv_Interno", "Advogadoadverso", "Nm_Empresa", "Denominacao",
    "classe_texto", "area_texto", "assunto_normalizado_texto", "assunto_texto", "fase_processual_texto",
    "status_texto", "status_tipo", "cidade", "uf", "vara_texto", "orgao_julgador_texto",
    "juiz_normalizado_texto", "tipo_de_processo_texto", "tipo_de_justica_texto",
    "tipo_de_requerente_texto", "tipo_de_requerido_texto",
])

normalized_categorical_features = []

for col in base_categorical_cols:
    new_col = f"fe_{clean_colname(col)}_norm"
    df_model[new_col] = df_model[col].apply(normalize_text)
    normalized_categorical_features.append(new_col)

normalized_categorical_features[:20], len(normalized_categorical_features)

## 10. Features combinadas

Combinações costumam ser muito fortes em jurimetria porque o risco raramente está em uma variável isolada.

Criamos:

- produto + ação;
- produto + assunto;
- ação + assunto;
- produto + ação + assunto;
- comarca + assunto;
- fase + assunto;
- escritório + assunto;
- UF + assunto.

In [ ]:
interaction_specs = [
    ("Nm_Produto", "Nm_Acao"),
    ("Nm_Produto", "Nm_Assunto"),
    ("Nm_Acao", "Nm_Assunto"),
    ("Nm_Produto", "Nm_Acao", "Nm_Assunto"),
    ("Uf", "Comarca"),
    ("Uf", "Nm_Assunto"),
    ("Comarca", "Nm_Assunto"),
    ("Fasedoprocesso", "Nm_Assunto"),
    ("Credenciado", "Nm_Assunto"),
    ("Adv_Interno", "Nm_Assunto"),
    ("classe_texto", "assunto_normalizado_texto"),
    ("fase_processual_texto", "assunto_normalizado_texto"),
]

interaction_features = []

for spec in interaction_specs:
    if all(c in df_model.columns for c in spec):
        new_col = "fe_inter_" + "__".join(clean_colname(c) for c in spec)
        df_model[new_col] = (
            df_model[list(spec)]
            .astype(str)
            .fillna("nao_informado")
            .agg(" | ".join, axis=1)
            .apply(normalize_text)
        )
        interaction_features.append(new_col)

interaction_features

## 11. Features de frequência e raridade

Criamos para cada categoria:

- frequência absoluta;
- frequência relativa;
- flag categoria rara.

Essas features devem ser calculadas apenas no treino em uma pipeline ideal. Aqui criamos uma versão exploratória para análise e baseline simples.

In [ ]:
frequency_features = []
freq_source_cols = normalized_categorical_features + interaction_features

for col in freq_source_cols:
    vc = df_model[col].value_counts(dropna=False)
    freq_abs_col = f"{col}_freq_abs"
    freq_pct_col = f"{col}_freq_pct"
    rare_col = f"{col}_is_rare"

    df_model[freq_abs_col] = df_model[col].map(vc).fillna(0)
    df_model[freq_pct_col] = df_model[freq_abs_col] / len(df_model)
    df_model[rare_col] = (df_model[freq_abs_col] < 30).astype(int)

    frequency_features += [freq_abs_col, freq_pct_col, rare_col]

frequency_features[:10], len(frequency_features)

## 12. Features históricas por grupo, sem usar target

Criamos agregações históricas **sem target**, usando apenas distribuição de valor por grupo:

- média de valor por produto;
- mediana de valor por assunto;
- p90 de valor por ação;
- quantidade de processos por grupo.

Essas features são menos arriscadas que target encoding, mas em produção também devem ser calculadas apenas com dados anteriores à data de previsão.

In [ ]:
group_agg_features = []

group_cols_for_value_stats = existing_cols(df_model, [
    "Nm_Produto", "Nm_Acao", "Nm_Assunto", "Comarca", "Uf", "Credenciado",
    "Fasedoprocesso", "classe_texto", "assunto_normalizado_texto",
])

for col in group_cols_for_value_stats:
    safe = clean_colname(col)

    stats = df_model.groupby(col, dropna=False)["fe_valor_ajuizado"].agg([
        "count", "mean", "median", lambda x: x.quantile(0.90)
    ])
    stats.columns = [
        f"fe_{safe}_qtd_historica",
        f"fe_{safe}_valor_medio_historico",
        f"fe_{safe}_valor_mediano_historico",
        f"fe_{safe}_valor_p90_historico",
    ]

    df_model = df_model.merge(stats.reset_index(), on=col, how="left")
    group_agg_features += list(stats.columns)

group_agg_features[:20], len(group_agg_features)

## 13. Features de texto consolidado

Mesmo sem usar textos longos de sentença/tutela, podemos criar um texto curto com campos disponíveis antes do desfecho:

- produto;
- ação;
- assunto;
- classe;
- assunto normalizado;
- fase;
- comarca;
- órgão julgador.

Isso permite usar `TfidfVectorizer` no pipeline.

In [ ]:
text_source_cols = existing_cols(df_model, [
    "Nm_Produto", "Nm_Acao", "Nm_Assunto", "Fasedoprocesso", "Comarca", "Uf",
    "classe_texto", "assunto_normalizado_texto", "assunto_texto", "fase_processual_texto",
    "cidade", "uf", "vara_texto", "orgao_julgador_texto", "tipo_de_processo_texto", "tipo_de_justica_texto",
])

if text_source_cols:
    df_model["fe_texto_curto_processo"] = (
        df_model[text_source_cols]
        .astype(str)
        .fillna("")
        .agg(" ".join, axis=1)
        .apply(lambda x: normalize_text(x) or "")
    )
else:
    df_model["fe_texto_curto_processo"] = ""

text_features = ["fe_texto_curto_processo"]
text_source_cols

## 14. Roteiro opcional para spaCy e HuggingFace

### spaCy

Pode ser útil para:

- extrair entidades de texto de eventos ou petições;
- normalizar termos jurídicos;
- criar contagem de entidades;
- detectar menções a dano moral, revisão contratual, negativação, fraude, cobrança etc.

### HuggingFace

Pode ser útil para gerar embeddings de textos jurídicos, como:

- assunto;
- movimentações;
- inicial;
- eventos processuais.

Cuidado: textos de sentença ou encerramento podem gerar leakage. Para este primeiro modelo, use apenas textos disponíveis antes da decisão.

In [ ]:
# Exemplo opcional com sentence-transformers / HuggingFace
# Rode apenas se a biblioteca estiver disponível e se o texto usado não tiver leakage.

# from sentence_transformers import SentenceTransformer
# model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
# texts = df_model["fe_texto_curto_processo"].fillna("").tolist()
# embeddings = model.encode(texts, batch_size=64, show_progress_bar=True)
# emb_cols = [f"fe_hf_emb_{i}" for i in range(embeddings.shape[1])]
# df_emb = pd.DataFrame(embeddings, columns=emb_cols, index=df_model.index)
# df_model = pd.concat([df_model, df_emb], axis=1)
# embedding_features = emb_cols

embedding_features = []

## 15. Feature sets por cenário

Criamos três cenários:

### Cenário A — baseline inicial seguro

Usa principalmente:

- valor;
- data inicial;
- produto/ação/assunto;
- território;
- DeepLegal estrutural não pós-resultado;
- texto curto.

### Cenário B — score atual da carteira

Inclui também:

- fase;
- escritório;
- advogado;
- passível de acordo;
- processo relevante;
- estimativa de perda;
- dias sem andamento.

Esse cenário pode performar melhor, mas precisa de validação temporal.

### Cenário C — CatBoost

Mantém categorias em formato bruto/normalizado e deixa o CatBoost lidar com elas.

In [ ]:
numeric_safe_features = existing_cols(df_model, [
    "fe_valor_ajuizado", "fe_valor_ajuizado_log1p", "fe_valor_ajuizado_winsor",
    "fe_valor_ajuizado_winsor_log1p", "fe_flag_valor_acima_p75",
    "fe_flag_valor_acima_p90", "fe_flag_valor_acima_p95", "fe_flag_valor_acima_p99",
] + date_features)

safe_category_source_names = [
    "Carteira", "Nm_Produto", "Nm_Acao", "Nm_Assunto", "Tipoprocesso", "Uf", "Comarca", "Nm_Empresa",
    "classe_texto", "area_texto", "assunto_normalizado_texto", "assunto_texto", "cidade", "uf", "vara_texto",
    "orgao_julgador_texto", "tipo_de_processo_texto", "tipo_de_justica_texto", "tipo_de_requerente_texto", "tipo_de_requerido_texto",
]

safe_categorical_features = []
for col in safe_category_source_names:
    norm_col = f"fe_{clean_colname(col)}_norm"
    if norm_col in df_model.columns:
        safe_categorical_features.append(norm_col)

safe_interaction_features = [c for c in interaction_features if not any(x in c for x in ["fasedoprocesso", "credenciado", "adv_interno"])]

management_categorical_source_names = [
    "Fasedoprocesso", "Credenciado", "Adv_Interno", "Advogadoadverso", "fase_processual_texto", "status_texto", "status_tipo",
]

management_categorical_features = []
for col in management_categorical_source_names:
    norm_col = f"fe_{clean_colname(col)}_norm"
    if norm_col in df_model.columns:
        management_categorical_features.append(norm_col)

numeric_engineered_features = existing_cols(df_model, frequency_features + group_agg_features + embedding_features)

scenario_a_features = list(dict.fromkeys(numeric_safe_features + safe_categorical_features + safe_interaction_features + text_features))
scenario_b_features = list(dict.fromkeys(scenario_a_features + management_features + management_categorical_features + interaction_features + numeric_engineered_features))
catboost_features = list(dict.fromkeys(numeric_safe_features + safe_categorical_features + safe_interaction_features + management_features + management_categorical_features + interaction_features + numeric_engineered_features))

feature_sets = {
    "scenario_a_baseline_seguro": scenario_a_features,
    "scenario_b_score_atual_com_cautela": scenario_b_features,
    "scenario_c_catboost": catboost_features,
}

feature_set_report = pd.DataFrame([{"cenario": k, "qtd_features": len(v)} for k, v in feature_sets.items()])
save_report(feature_set_report, "fe_feature_sets_summary.csv")
feature_set_report

## 16. Remover qualquer leakage remanescente

In [ ]:
for scenario, feats in feature_sets.items():
    clean_feats = []
    removed = []

    for f in feats:
        if is_leakage_col(f):
            removed.append(f)
        elif f not in df_model.columns:
            removed.append(f)
        else:
            clean_feats.append(f)

    feature_sets[scenario] = clean_feats

    print(scenario)
    print("Features finais:", len(clean_feats))
    print("Removidas:", len(removed))
    if removed:
        print(removed[:20])
    print("-" * 80)

## 17. Separar treino e teste temporal

Evite split aleatório se possível. Como estamos lidando com processos jurídicos, o ideal é usar split temporal.

Aqui usamos `Datainicial`:

- treino: processos mais antigos;
- teste: processos mais recentes.

In [ ]:
if "Datainicial" not in df_model.columns:
    raise KeyError("Datainicial não encontrada. Para o primeiro modelo, prefira split temporal.")

df_model = df_model.sort_values("Datainicial").copy()
split_date = df_model["Datainicial"].quantile(0.80)

train_mask = df_model["Datainicial"] <= split_date
test_mask = df_model["Datainicial"] > split_date

df_train = df_model[train_mask].copy()
df_test = df_model[test_mask].copy()

split_report = pd.DataFrame([{
    "split_date": split_date,
    "qtd_train": len(df_train),
    "qtd_test": len(df_test),
    "taxa_target_train": df_train[TARGET_COL].mean(),
    "taxa_target_test": df_test[TARGET_COL].mean(),
    "valor_medio_train": df_train["fe_valor_ajuizado"].mean(),
    "valor_medio_test": df_test["fe_valor_ajuizado"].mean(),
}])

save_report(split_report, "fe_temporal_split_report.csv")
split_report

## 18. Pipeline sklearn — baseline seguro

Este pipeline usa:

- imputação numérica;
- padronização numérica;
- one-hot encoding para categóricas;
- TF-IDF para texto curto;
- regressão logística como baseline.

A ideia não é buscar o melhor modelo agora, mas validar se as features têm sinal preditivo.

In [ ]:
scenario = "scenario_a_baseline_seguro"
features = feature_sets[scenario]

X_train = df_train[features].copy()
y_train = df_train[TARGET_COL].copy()
X_test = df_test[features].copy()
y_test = df_test[TARGET_COL].copy()

numeric_cols = [c for c in features if c in df_model.columns and pd.api.types.is_numeric_dtype(df_model[c])]
text_cols = [c for c in text_features if c in features]
categorical_cols = [c for c in features if c not in numeric_cols and c not in text_cols]

print("Numeric:", len(numeric_cols))
print("Categorical:", len(categorical_cols))
print("Text:", len(text_cols))

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="nao_informado")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20))
])

transformers = []

if numeric_cols:
    transformers.append(("num", numeric_pipeline, numeric_cols))

if categorical_cols:
    transformers.append(("cat", categorical_pipeline, categorical_cols))

if text_cols:
    text_col = text_cols[0]
    text_pipeline = Pipeline(steps=[
        ("selector", FunctionTransformer(lambda x: x[text_col].fillna("").astype(str), validate=False)),
        ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=10))
    ])
    transformers.append(("txt", text_pipeline, [text_col]))

preprocessor = ColumnTransformer(transformers=transformers, remainder="drop", sparse_threshold=0.3)

baseline_model = LogisticRegression(max_iter=1000, class_weight="balanced", solver="saga", n_jobs=-1, random_state=RANDOM_STATE)

baseline_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", baseline_model)
])

baseline_pipeline

## 19. Treinar baseline rápido

In [ ]:
baseline_pipeline.fit(X_train, y_train)

proba_test = baseline_pipeline.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= 0.5).astype(int)

roc = roc_auc_score(y_test, proba_test)
pr_auc = average_precision_score(y_test, proba_test)

print("ROC AUC:", roc)
print("PR AUC:", pr_auc)
print(classification_report(y_test, pred_test))

baseline_metrics = pd.DataFrame([{
    "cenario": scenario,
    "modelo": "LogisticRegression_baseline",
    "roc_auc": roc,
    "pr_auc": pr_auc,
    "qtd_train": len(X_train),
    "qtd_test": len(X_test),
    "qtd_features_brutas": len(features),
}])

save_report(baseline_metrics, "fe_baseline_logistic_metrics.csv")
baseline_metrics

## 20. Pipeline com category_encoders

Opcional.

`category_encoders` pode ser útil para:

- TargetEncoder;
- CatBoostEncoder;
- JamesSteinEncoder;
- LeaveOneOutEncoder.

Cuidado: encoders supervisionados precisam estar dentro do pipeline e fitados apenas no treino.

In [ ]:
try:
    import category_encoders as ce

    high_card_cols = [c for c in categorical_cols if df_train[c].nunique(dropna=True) > 30]
    low_card_cols = [c for c in categorical_cols if c not in high_card_cols]

    transformers_ce = []
    if numeric_cols:
        transformers_ce.append(("num", numeric_pipeline, numeric_cols))
    if low_card_cols:
        transformers_ce.append(("cat_low", categorical_pipeline, low_card_cols))
    if high_card_cols:
        high_card_pipeline = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value="nao_informado")),
            ("target_encoder", ce.CatBoostEncoder(cols=high_card_cols, random_state=RANDOM_STATE))
        ])
        transformers_ce.append(("cat_high", high_card_pipeline, high_card_cols))
    if text_cols:
        text_col = text_cols[0]
        text_pipeline = Pipeline(steps=[
            ("selector", FunctionTransformer(lambda x: x[text_col].fillna("").astype(str), validate=False)),
            ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=10))
        ])
        transformers_ce.append(("txt", text_pipeline, [text_col]))

    preprocessor_ce = ColumnTransformer(transformers=transformers_ce, remainder="drop", sparse_threshold=0.3)

    ce_pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor_ce),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced", solver="saga", n_jobs=-1, random_state=RANDOM_STATE))
    ])

    ce_pipeline.fit(X_train, y_train)
    proba_ce = ce_pipeline.predict_proba(X_test)[:, 1]

    ce_metrics = pd.DataFrame([{
        "cenario": scenario,
        "modelo": "LogisticRegression_CatBoostEncoder",
        "roc_auc": roc_auc_score(y_test, proba_ce),
        "pr_auc": average_precision_score(y_test, proba_ce),
        "qtd_high_card_cols": len(high_card_cols),
        "qtd_low_card_cols": len(low_card_cols),
    }])

    save_report(ce_metrics, "fe_category_encoders_metrics.csv")
    display(ce_metrics)

except ImportError:
    print("category_encoders não instalado. Pule esta etapa ou instale com: pip install category_encoders")

## 21. Pipeline com skrub / dirty-cat

Opcional.

Essas bibliotecas são úteis para categorias sujas e de alta cardinalidade, como:

- comarcas;
- varas;
- órgãos julgadores;
- escritórios;
- advogados;
- nomes de partes.

Ideia de uso:

- SimilarityEncoder;
- GapEncoder;
- MinHashEncoder.

In [ ]:
# Exemplo conceitual:
# try:
#     from skrub import GapEncoder
#     dirty_cat_cols = [c for c in categorical_cols if df_train[c].nunique(dropna=True) > 50]
#     gap_pipeline = Pipeline(steps=[
#         ("imputer", SimpleImputer(strategy="constant", fill_value="nao_informado")),
#         ("gap", GapEncoder(n_components=20))
#     ])
# except ImportError:
#     print("skrub não instalado. Avalie instalar se houver muitas categorias textuais sujas.")
#
# Se estiver usando dirty-cat em vez de skrub:
# try:
#     from dirty_cat import SimilarityEncoder
# except ImportError:
#     print("dirty-cat não instalado.")

## 22. Pipeline CatBoost

CatBoost costuma ser uma excelente opção para dados jurídicos tabulares com muitas categorias.

Vantagens:

- lida bem com variáveis categóricas;
- reduz necessidade de one-hot;
- suporta alta cardinalidade;
- funciona bem como primeiro modelo forte.

In [ ]:
try:
    from catboost import CatBoostClassifier, Pool

    scenario_cb = "scenario_c_catboost"
    cb_features = feature_sets[scenario_cb]

    X_train_cb = df_train[cb_features].copy()
    X_test_cb = df_test[cb_features].copy()

    cat_features_cb = [c for c in cb_features if not pd.api.types.is_numeric_dtype(df_model[c])]

    for col in cat_features_cb:
        X_train_cb[col] = X_train_cb[col].astype(str).fillna("nao_informado")
        X_test_cb[col] = X_test_cb[col].astype(str).fillna("nao_informado")

    numeric_cb = [c for c in cb_features if c not in cat_features_cb]
    for col in numeric_cb:
        X_train_cb[col] = pd.to_numeric(X_train_cb[col], errors="coerce")
        X_test_cb[col] = pd.to_numeric(X_test_cb[col], errors="coerce")

    train_pool = Pool(X_train_cb, y_train, cat_features=cat_features_cb)
    test_pool = Pool(X_test_cb, y_test, cat_features=cat_features_cb)

    cb_model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=RANDOM_STATE,
        auto_class_weights="Balanced",
        verbose=100
    )

    cb_model.fit(train_pool, eval_set=test_pool, use_best_model=True)
    proba_cb = cb_model.predict_proba(test_pool)[:, 1]

    cb_metrics = pd.DataFrame([{
        "cenario": scenario_cb,
        "modelo": "CatBoostClassifier",
        "roc_auc": roc_auc_score(y_test, proba_cb),
        "pr_auc": average_precision_score(y_test, proba_cb),
        "qtd_features": len(cb_features),
        "qtd_cat_features": len(cat_features_cb),
    }])

    save_report(cb_metrics, "fe_catboost_metrics.csv")
    display(cb_metrics)

    feature_importance = pd.DataFrame({
        "feature": cb_features,
        "importance": cb_model.get_feature_importance(train_pool)
    }).sort_values("importance", ascending=False)

    save_report(feature_importance, "fe_catboost_feature_importance.csv")
    display(feature_importance.head(50))

except ImportError:
    print("catboost não instalado. Instale com: pip install catboost")

## 23. Feature-engine

Opcional.

`feature-engine` pode ser útil para criar pipelines mais robustos com:

- imputação categórica;
- rare label encoding;
- discretização;
- winsorização;
- transformação logarítmica;
- seleção de features.

Recomendação:

- usar `RareLabelEncoder` para categorias raras;
- usar `Winsorizer` para valores extremos;
- usar discretização para variáveis financeiras.

In [ ]:
# Exemplo opcional:
# try:
#     from feature_engine.encoding import RareLabelEncoder
#     from feature_engine.outliers import Winsorizer
#     from feature_engine.discretisation import EqualFrequencyDiscretiser
#
#     rare_encoder = RareLabelEncoder(tol=0.01, n_categories=10, variables=categorical_cols)
#
# except ImportError:
#     print("feature-engine não instalado. Instale com: pip install feature-engine")

## 24. OptBinning

Opcional.

`optbinning` é útil em contexto bancário para:

- variáveis financeiras;
- criação de bins monotônicos;
- scorecards;
- interpretabilidade;
- Weight of Evidence.

Cuidado: o binning supervisionado usa o target, então deve ser fitado apenas no treino.

In [ ]:
# Exemplo opcional:
# try:
#     from optbinning import OptimalBinning
#
#     variable = "fe_valor_ajuizado_log1p"
#     optb = OptimalBinning(name=variable, dtype="numerical", solver="cp")
#     optb.fit(df_train[variable].values, y_train.values)
#
#     df_train[f"{variable}_optbin"] = optb.transform(df_train[variable].values, metric="bins")
#     df_test[f"{variable}_optbin"] = optb.transform(df_test[variable].values, metric="bins")
#
#     display(optb.binning_table.build())
#
# except ImportError:
#     print("optbinning não instalado. Instale com: pip install optbinning")

## 25. tsfresh

`tsfresh` só faz sentido se você tiver a base de eventos por processo com sequência temporal.

Exemplos de features que poderiam ser extraídas dos eventos:

- quantidade de eventos por mês;
- tempo entre eventos;
- aceleração de movimentações;
- períodos sem movimentação;
- frequência de tipos de evento;
- tendência temporal de movimentação;
- entropia dos tipos de evento.

In [ ]:
# Exemplo conceitual para quando houver DeepLegal Eventos:
#
# from tsfresh import extract_features
# eventos_numeric = df_eventos.copy()
# eventos_numeric["value"] = 1
# eventos_numeric["time"] = pd.to_datetime(eventos_numeric["data"])
#
# features_tsfresh = extract_features(
#     eventos_numeric,
#     column_id="numero_processo",
#     column_sort="time",
#     column_value="value"
# )
#
# Depois juntar com a ABT por numero_processo.

## 26. Salvar ABTs de features

Salvamos:

1. ABT completa com todas as features criadas.
2. ABT cenário A: baseline seguro.
3. ABT cenário B: score atual com cautela.
4. ABT cenário C: CatBoost.

In [ ]:
abt_full_path = PROCESSED_DIR / "abt_features_target_banco_ganhou_full.parquet"
df_model.to_parquet(abt_full_path, index=False)

for scenario_name, features in feature_sets.items():
    cols_to_save = existing_cols(df_model, features + [TARGET_COL, "Datainicial"])
    out = df_model[cols_to_save].copy()
    path = PROCESSED_DIR / f"abt_features_{scenario_name}.parquet"
    out.to_parquet(path, index=False)
    print(scenario_name, "->", path, out.shape)

feature_dictionary_rows = []

for scenario_name, features in feature_sets.items():
    for f in features:
        if f in numeric_safe_features:
            tipo = "numeric_safe"
        elif f in safe_categorical_features:
            tipo = "categorical_safe"
        elif f in safe_interaction_features:
            tipo = "interaction_safe"
        elif f in management_features:
            tipo = "management_caution"
        elif f in management_categorical_features:
            tipo = "management_categorical_caution"
        elif f in numeric_engineered_features:
            tipo = "engineered_frequency_or_group_stats_caution"
        elif f in text_features:
            tipo = "text_tfidf"
        else:
            tipo = "other"

        feature_dictionary_rows.append({
            "cenario": scenario_name,
            "feature": f,
            "tipo": tipo,
            "dtype": str(df_model[f].dtype) if f in df_model.columns else None,
            "nunique": df_model[f].nunique(dropna=True) if f in df_model.columns else None,
            "missing_rate": df_model[f].isna().mean() if f in df_model.columns else None,
        })

feature_dictionary = pd.DataFrame(feature_dictionary_rows)
save_report(feature_dictionary, "fe_feature_dictionary.csv")

print("ABT completa:", abt_full_path)
feature_dictionary.head(50)

## 27. Checklist de validação antes da modelagem oficial

### Target

- A regra de `target_banco_ganhou` foi aprovada pelo jurídico?
- Acordos devem ser perda, ganho ou categoria separada?
- Extinção sem mérito deve ser sempre êxito?
- Prescrição/decadência devem ser êxito?
- Limpeza de base e baixa administrativa devem ser removidas?

### Leakage

Remover obrigatoriamente:

- `Motivo_Encerramento`;
- `Situacao`;
- `Dataencerramento`;
- `desfecho_categoria`;
- qualquer resultado/sentença/condenação/acordo;
- textos posteriores ao julgamento;
- predições externas que já usem resultado processual.

### Split

Preferir split temporal:

- treino em processos antigos;
- teste em processos mais recentes.

### Avaliação

Não olhar só ROC AUC. Avaliar também:

- PR AUC;
- recall da classe perda;
- precision da classe perda;
- lift no top 5%, 10%, 20%;
- valor financeiro capturado no top k%;
- calibração;
- estabilidade por ano;
- estabilidade por produto/assunto.

### Produto final

O primeiro modelo deve entregar:

- probabilidade de perda;
- ranking de processos;
- explicabilidade das principais features;
- análise de ganho financeiro;
- limiares operacionais para priorização.

# Próximo passo recomendado

Depois deste notebook, o próximo notebook deveria ser:

```text
07_modelagem_baseline_target_banco_ganhou.ipynb
```

Ele deve:

1. treinar modelos baseline;
2. comparar Logistic Regression, CatBoost e talvez LightGBM/XGBoost;
3. avaliar por split temporal;
4. calibrar probabilidades;
5. analisar lift financeiro;
6. gerar ranking de processos;
7. produzir explicabilidade global e local.